In [1]:
import sqlite3
import pandas as pd
from scipy import stats

In [2]:
DB_PATH = "bosch_dw.db"

rating_cols = [
    "rating",
    "rating_programs",
    "rating_loading",
    "rating_washing",
    "rating_noise",
]

RATING_DICT = {
    "rating": "Overall Rating",
    "rating_programs": "Programs Rating",
    "rating_loading": "Loading Rating",
    "rating_washing": "Washing Rating",
    "rating_noise": "Noise Rating"
}

ratings_sql = ", ".join(rating_cols)

with sqlite3.connect(DB_PATH) as conn:
    cols = pd.read_sql("PRAGMA table_info(product_db);", conn)

    claim_cols = cols.loc[
        cols["name"].str.startswith("has_") &
        (cols["name"] != "has_dosage_assist"),
        "name"
    ]

    all_data = {}
    for col in claim_cols:
        df = pd.read_sql(f"""
            SELECT {col}, {ratings_sql}
            FROM review_db JOIN product_db ON product_db.product_id = review_db.product_id
            WHERE {col} IS NOT NULL
        """, conn)
        df[col] = df[col].astype(bool)
        all_data[col] = df

In [3]:
records = []

for col, df in all_data.items():
    claim_name = col[4:].replace("_", " ").title()
    for rcol in rating_cols:
        sub = df[[col, rcol]].dropna(subset=[rcol])
        true_sub  = sub[sub[col]][rcol]
        false_sub = sub[~sub[col]][rcol]

        if true_sub.empty or false_sub.empty:
            continue

        pct_neg_true  = (true_sub  <= 3).mean() * 100
        pct_neg_false = (false_sub <= 3).mean() * 100
        delta_neg     = pct_neg_true - pct_neg_false

        mean_true  = true_sub.mean()
        mean_false = false_sub.mean()
        delta_mean = mean_true - mean_false

        u_stat, p_value = stats.mannwhitneyu(true_sub, false_sub, alternative="two-sided")

        records.append({
            "claim":        claim_name,
            "rating":       RATING_DICT[rcol],
            "n_true":       len(true_sub),
            "n_false":      len(false_sub),
            "pct_neg_true":  round(pct_neg_true,  2),
            "pct_neg_false": round(pct_neg_false, 2),
            "delta_neg":     round(delta_neg,  2),
            "mean_true":     round(mean_true,  3),
            "mean_false":    round(mean_false, 3),
            "delta_mean":    round(delta_mean, 3),
            "p_value":       round(p_value, 3),
            "significant":   p_value < 0.05,
        })

delta_df = pd.DataFrame(records)

In [4]:
delta_df.sort_values(by="delta_neg",ascending=True).head(10)

,claim,rating,n_true,n_false,pct_neg_true,pct_neg_false,delta_neg,mean_true,mean_false,delta_mean,p_value,significant
80,Smart Start,Overall Rating,13125,2146,2.10,3.54,-1.45,4.849,4.808,0.040,0.062,False
20,Favorite Program,Overall Rating,13443,1828,2.13,3.56,-1.43,4.846,4.818,0.029,0.697,False
60,Programme Download,Overall Rating,13588,1683,2.15,3.51,-1.36,4.845,4.825,0.021,0.625,False
30,Home Connect,Overall Rating,13588,1683,2.15,3.51,-1.36,4.845,4.825,0.021,0.625,False
70,Remote Diagnostics,Overall Rating,13588,1683,2.15,3.51,-1.36,4.845,4.825,0.021,0.625,False
100,Wash Assistant,Overall Rating,13588,1683,2.15,3.51,-1.36,4.845,4.825,0.021,0.625,False
35,Hygiene Certified,Overall Rating,13588,1683,2.15,3.51,-1.36,4.845,4.825,0.021,0.625,False
24,Favorite Program,Noise Rating,12330,1602,2.32,3.43,-1.11,4.797,4.753,0.043,0.085,False
91,Time Light,Programs Rating,2650,11364,0.91,1.98,-1.07,4.913,4.806,0.107,0.000,True
92,Time Light,Loading Rating,2647,11329,1.40,2.45,-1.06,4.864,4.786,0.078,0.000,True


In [5]:
delta_df.sort_values(by="delta_neg",ascending=False).head(10)

,claim,rating,n_true,n_false,pct_neg_true,pct_neg_false,delta_neg,mean_true,mean_false,delta_mean,p_value,significant
3,Aquastop,Washing Rating,11281,2686,2.04,1.27,0.77,4.821,4.890,-0.069,0.000,True
85,Status Light,Overall Rating,471,14800,2.97,2.28,0.70,4.786,4.845,-0.059,0.001,True
4,Aquastop,Noise Rating,11251,2681,2.58,1.90,0.68,4.786,4.816,-0.030,0.018,True
54,Open Assist,Noise Rating,272,13660,2.94,2.44,0.50,4.812,4.791,0.021,0.240,False
99,Vario Hinge,Noise Rating,722,13210,2.91,2.42,0.49,4.738,4.795,-0.056,0.001,True
1,Aquastop,Programs Rating,11313,2701,1.87,1.41,0.46,4.814,4.881,-0.067,0.000,True
88,Status Light,Washing Rating,430,13537,2.33,1.88,0.45,4.747,4.837,-0.091,0.000,True
49,Max Flex Pro,Noise Rating,1123,12809,2.85,2.41,0.44,4.790,4.792,-0.002,1.000,False
65,Rackmatic,Overall Rating,698,14573,2.72,2.28,0.44,4.801,4.845,-0.044,0.002,True
0,Aquastop,Overall Rating,12397,2874,2.38,1.95,0.43,4.834,4.882,-0.048,0.000,True


In [6]:
delta_df.sort_values(by="delta_mean",ascending=True).head(10)

,claim,rating,n_true,n_false,pct_neg_true,pct_neg_false,delta_neg,mean_true,mean_false,delta_mean,p_value,significant
88,Status Light,Washing Rating,430,13537,2.33,1.88,0.45,4.747,4.837,-0.091,0.000,True
98,Vario Hinge,Washing Rating,722,13245,1.94,1.89,0.05,4.760,4.838,-0.078,0.000,True
96,Vario Hinge,Programs Rating,725,13289,2.07,1.76,0.31,4.759,4.830,-0.072,0.000,True
3,Aquastop,Washing Rating,11281,2686,2.04,1.27,0.77,4.821,4.890,-0.069,0.000,True
78,Silent Power Drive,Washing Rating,13245,722,1.91,1.52,0.39,4.831,4.899,-0.068,0.000,True
1,Aquastop,Programs Rating,11313,2701,1.87,1.41,0.46,4.814,4.881,-0.067,0.000,True
86,Status Light,Programs Rating,432,13582,1.85,1.77,0.08,4.764,4.829,-0.065,0.000,True
97,Vario Hinge,Loading Rating,724,13252,2.49,2.24,0.25,4.743,4.804,-0.060,0.000,True
85,Status Light,Overall Rating,471,14800,2.97,2.28,0.70,4.786,4.845,-0.059,0.001,True
99,Vario Hinge,Noise Rating,722,13210,2.91,2.42,0.49,4.738,4.795,-0.056,0.001,True


In [7]:
delta_df.sort_values(by="delta_mean",ascending=False).head(20)

,claim,rating,n_true,n_false,pct_neg_true,pct_neg_false,delta_neg,mean_true,mean_false,delta_mean,p_value,significant
91,Time Light,Programs Rating,2650,11364,0.91,1.98,-1.07,4.913,4.806,0.107,0.000,True
16,Extra Clean Zone,Programs Rating,3124,10890,1.15,1.96,-0.80,4.900,4.806,0.095,0.000,True
93,Time Light,Washing Rating,2638,11329,1.06,2.08,-1.02,4.911,4.817,0.094,0.000,True
11,Emotion Light,Programs Rating,2538,11476,1.22,1.90,-0.68,4.901,4.810,0.091,0.000,True
13,Emotion Light,Washing Rating,2524,11443,1.19,2.04,-0.86,4.906,4.819,0.087,0.000,True
18,Extra Clean Zone,Washing Rating,3108,10859,1.22,2.08,-0.86,4.900,4.816,0.084,0.000,True
41,Intelligent Prog,Programs Rating,2180,11834,1.38,1.85,-0.47,4.893,4.815,0.078,0.000,True
92,Time Light,Loading Rating,2647,11329,1.40,2.45,-1.06,4.864,4.786,0.078,0.000,True
43,Intelligent Prog,Washing Rating,2168,11799,1.25,2.01,-0.76,4.899,4.822,0.077,0.000,True
90,Time Light,Overall Rating,2813,12458,1.81,2.41,-0.60,4.905,4.829,0.076,0.000,True
